<a href="https://colab.research.google.com/github/14marcos1/curso_colab_2026/blob/main/acre_2010_2023_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GEMINI:

Atue como um especialista em engenharia de dados de saúde pública e crie um script em Python para o Google Colab para extrair dados do Sistema de Informações sobre Mortalidade (SIM/DATASUS).

Siga RIGOROSAMENTE os parâmetros e as regras de arquitetura fornecidas.

PARÂMETROS DA PESQUISA (Ajuste se necessário):

Estado (UF): [AC]
Período: [2010 a 2023]
Grupo 1: "Neoplasias" -> CIDs-10 iniciados com as letras ['C', 'D']
Grupo 2: "Aparelho Circulatório" -> CIDs-10 iniciados com a letra ['I']
REGRAS OBRIGATÓRIAS DE ARQUITETURA DO CÓDIGO (NÃO MUDAR OU OMITIR):

INSTALAÇÃO DE DEPENDÊNCIAS DE SISTEMA (LINUX E PYTHON): Instale no início do script os pacotes de sistema necessários para converter os arquivos do DATASUS: !apt-get update -qq && !apt-get install -y -qq p7zip-full !pip install pyreaddbc dbfread pandas -q

IMPORTAÇÃO E SINTAXE OBRIGATÓRIA DO DBC2DBF (CRÍTICO):

Importe a função usando exatamente: from pyreaddbc import dbc2dbf
Para converter os arquivos, passe estritamente DUAS strings com os caminhos completos de entrada e saída: dbc2dbf(caminho_dbc, caminho_dbf)
Nunca use dbc2dbf.dbc2dbf() nem métodos sem o parâmetro do arquivo de saída.
BAIXAMENTO E CONVERSÃO (NÃO USE PYSUS ONLINE):

Baixe os arquivos do FTP oficial: "ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ANO}.DBC"
Adicione tentativa de download com extensão minúscula (.dbc) caso a maiúscula falhe.
Leia o arquivo .dbf convertido com dbfread.DBF(..., encoding="latin1").
Remova os arquivos temporários (.dbc e .dbf) do disco a cada loop para não lotar a memória do Colab.
TRATAMENTO DA COLUNA E AGRUPAMENTO:

Filtre a coluna 'CAUSABAS' utilizando os prefixos de CID fornecidos nos parâmetros.
Conte o total de óbitos ano a ano para cada grupo ("Neoplasias" e "Aparelho Circulatório").
PADRONIZAÇÃO RÍGIDA DA SAÍDA:

O arquivo de saída DEVE ser salvo OBRIGATORIAMENTE no formato: f"mortalidade_{UF.lower()}_consolidado.csv" (Exemplo para o Acre: "mortalidade_ac_consolidado.csv").
A planilha DEVE conter exatamente 3 colunas com estes nomes em maiúsculo: ANO, GRUPO, OBITOS.
Inclua no final o comando google.colab.files.download() para baixar a planilha no computador.
Entregue o código completo em um ÚNICO bloco executável.



In [1]:
# Instalação de dependências de sistema e pacotes Python necessários
!apt-get update -qq && !apt-get install -y -qq p7zip-full
!pip install pyreaddbc dbfread pandas -q

import os
import urllib.request
import pandas as pd
from dbfread import DBF
from pyreaddbc import dbc2dbf
from google.colab import files

# Parâmetros da pesquisa
UF = "AC"
ANOS = list(range(2010, 2024))  # 2010 a 2023
GRUPOS = {
    "Neoplasias": ("C", "D"),
    "Aparelho Circulatório": ("I",)
}

resultados = []

for ano in ANOS:
    dbc_file = f"DO{UF}{ano}.DBC"
    dbf_file = f"DO{UF}{ano}.dbf"

    url_base = f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ano}"
    urls_to_try = [f"{url_base}.DBC", f"{url_base}.dbc"]

    downloaded = False
    for url in urls_to_try:
        try:
            print(f"Baixando dados do ano {ano} ({url})...")
            urllib.request.urlretrieve(url, dbc_file)
            downloaded = True
            break
        except Exception:
            continue

    if not downloaded:
        print(f"Erro: Não foi possível realizar o download para o ano {ano}.")
        continue

    try:
        # Conversão estrita DBC -> DBF
        dbc2dbf(dbc_file, dbf_file)

        # Leitura da tabela DBF
        table = DBF(dbf_file, encoding="latin1", load=True)
        df = pd.DataFrame(iter(table))

        if "CAUSABAS" in df.columns:
            # Tratamento da coluna de CIDs
            causabas = df["CAUSABAS"].astype(str).str.strip().str.upper()

            # Contagem por grupo
            qtd_neoplasias = causabas.str.startswith(GRUPOS["Neoplasias"]).sum()
            resultados.append({"ANO": ano, "GRUPO": "Neoplasias", "OBITOS": qtd_neoplasias})

            qtd_circulatorio = causabas.str.startswith(GRUPOS["Aparelho Circulatório"]).sum()
            resultados.append({"ANO": ano, "GRUPO": "Aparelho Circulatório", "OBITOS": qtd_circulatorio})

            print(f"Ano {ano} processado com sucesso.")
        else:
            print(f"Aviso: Coluna 'CAUSABAS' ausente no ano {ano}.")

    except Exception as e:
        print(f"Erro no processamento do ano {ano}: {e}")

    finally:
        # Remoção dos arquivos temporários a cada ciclo para otimizar espaço em disco
        if os.path.exists(dbc_file):
            os.remove(dbc_file)
        if os.path.exists(dbf_file):
            os.remove(dbf_file)

# Consolidação final do DataFrame
df_final = pd.DataFrame(resultados)

# Exportação padronizada do CSV
nome_arquivo_saida = f"mortalidade_{UF.lower()}_consolidado.csv"
df_final.to_csv(nome_arquivo_saida, index=False)

print(f"\nProcessamento finalizado. Iniciando download do arquivo '{nome_arquivo_saida}'...")

# Download automático para o ambiente local
files.download(nome_arquivo_saida)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/bin/bash: line 1: !apt-get: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.0 MB/s eta 0:00:00
Baixando dados do ano 2010 (ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2010.DBC)...
Ano 2010 processado com sucesso.
Baixando dados do ano 2011 (ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2011.DBC)...
Ano 2011 processado com sucesso.
Baixando dados do ano 2012 (ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2012.DBC)...
Ano 2012 processado com sucesso.
Baixando dados do ano 2013 (ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2013.DBC)...
Ano 2013 processado com sucesso.
Baixando dados do ano 2014 (ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2014.DBC)...
Ano 2014 processado com sucesso.
B

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

GERANDO O GRAFICO



código

Atue como um especialista em engenharia de dados de saúde pública e crie um script em Python para o Google Colab para extrair dados do Sistema de Informações sobre Mortalidade (SIM/DATASUS).

Siga RIGOROSAMENTE os parâmetros e as regras de arquitetura fornecidas.

---
PARÂMETROS DA PESQUISA (Ajuste se necessário):
- Estado (UF): [AC]
- Período: [2010 a 2023]
- Grupo 1: "Neoplasias" -> CIDs-10 iniciados com as letras ['C', 'D']
- Grupo 2: "Aparelho Circulatório" -> CIDs-10 iniciados com a letra ['I']
---

REGRAS OBRIGATÓRIAS DE ARQUITETURA DO CÓDIGO (NÃO MUDAR OU OMITIR):

1. INSTALAÇÃO DE DEPENDÊNCIAS DE SISTEMA (LINUX E PYTHON):
   Instale no início do script os pacotes de sistema necessários para converter os arquivos do DATASUS:
   !apt-get update -qq && !apt-get install -y -qq p7zip-full
   !pip install pyreaddbc dbfread pandas -q

2. IMPORTAÇÃO E SINTAXE OBRIGATÓRIA DO DBC2DBF (CRÍTICO):
   - Importe a função usando exatamente: `from pyreaddbc import dbc2dbf`
   - Para converter os arquivos, passe estritamente DUAS strings com os caminhos completos de entrada e saída: `dbc2dbf(caminho_dbc, caminho_dbf)`
   - Nunca use `dbc2dbf.dbc2dbf()` nem métodos sem o parâmetro do arquivo de saída.

3. BAIXAMENTO E CONVERSÃO (NÃO USE PYSUS ONLINE):
   - Baixe os arquivos do FTP oficial: "ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ANO}.DBC"
   - Adicione tentativa de download com extensão minúscula (.dbc) caso a maiúscula falhe.
   - Leia o arquivo .dbf convertido com `dbfread.DBF(..., encoding="latin1")`.
   - Remova os arquivos temporários (.dbc e .dbf) do disco a cada loop para não lotar a memória do Colab.

4. TRATAMENTO DA COLUNA E AGRUPAMENTO:
   - Filtre a coluna 'CAUSABAS' utilizando os prefixos de CID fornecidos nos parâmetros.
   - Conte o total de óbitos ano a ano para cada grupo ("Neoplasias" e "Aparelho Circulatório").

5. PADRONIZAÇÃO RÍGIDA DA SAÍDA:
   - O arquivo de saída DEVE ser salvo OBRIGATORIAMENTE no formato: f"mortalidade_{UF.lower()}_consolidado.csv" (Exemplo para o Acre: "mortalidade_ac_consolidado.csv").
   - A planilha DEVE conter exatamente 3 colunas com estes nomes em maiúsculo: ANO, GRUPO, OBITOS.
   - Inclua no final o comando `google.colab.files.download()` para baixar a planilha no computador.
   - Entregue o código completo em um ÚNICO bloco executável.

In [2]:
# Instalação e importação das bibliotecas necessárias
!pip install plotly pandas -q

import os
import glob
import pandas as pd
import plotly.express as px
from google.colab import files

# 1. Localização flexível do arquivo CSV no ambiente
arquivo_alvo = None
arquivos_encontrados = glob.glob("*_consolidado.csv")

if arquivos_encontrados:
    arquivo_alvo = arquivos_encontrados[0]
elif os.path.exists("mortalidade_ac_consolidado.csv"):
    arquivo_alvo = "mortalidade_ac_consolidado.csv"

# Checagem de segurança
if not arquivo_alvo:
    print("ERRO: Nenhum arquivo CSV consolidado foi encontrado na pasta atual.")
    print("Por favor, certifique-se de executar o script da Etapa 1 ou faça o upload do arquivo 'mortalidade_ac_consolidado.csv'.")
else:
    print(f"Lendo dados do arquivo: '{arquivo_alvo}'...")
    df = pd.read_csv(arquivo_alvo)

    # 2. Tratamento e padronização dos dados
    df['ANO'] = pd.to_numeric(df['ANO'], errors='coerce')
    df['OBITOS'] = pd.to_numeric(df['OBITOS'], errors='coerce')
    df = df.dropna(subset=['ANO', 'OBITOS'])
    df['ANO'] = df['ANO'].astype(int)

    # Identificação do estado e intervalo temporal para o título
    uf_extracao = arquivo_alvo.split('_')[1].upper() if '_' in arquivo_alvo else "AC"
    ano_inicio = df['ANO'].min()
    ano_fim = df['ANO'].max()

    # 3. Criação do gráfico interativo
    fig = px.line(
        df,
        x="ANO",
        y="OBITOS",
        color="GRUPO",
        markers=True,
        title=f"Evolução Temporal da Mortalidade por Grupo de Causa GEMINI ({uf_extracao}) - {ano_inicio} a {ano_fim}",
        labels={
            "ANO": "Ano de Ocorrência",
            "OBITOS": "Número de Óbitos",
            "GRUPO": "Grupo de Causas"
        },
        template="plotly_white"
    )

    # Ajustes finos de layout e formatação dos eixos
    fig.update_layout(
        hovermode="x unified",
        title_font=dict(size=18, family="Arial"),
        xaxis=dict(
            dtick=1,  # Força o intervalo ano a ano sem decimais
            tickmode="linear",
            showgrid=True
        ),
        yaxis=dict(
            showgrid=True,
            zeroline=True
        ),
        legend=dict(
            title_text="Grupo CID-10",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.update_traces(hovertemplate="%{y:,} óbitos")

    # 4. Exibição no notebook
    fig.show()

    # 5. Salvamento e download do arquivo HTML interativo
    arquivo_html = "mortalidade_grafico_interativo.html"
    fig.write_html(arquivo_html)
    print(f"\nGráfico salvo em '{arquivo_html}'. Iniciando o download...")
    files.download(arquivo_html)

Lendo dados do arquivo: 'mortalidade_ac_consolidado.csv'...



Gráfico salvo em 'mortalidade_grafico_interativo.html'. Iniciando o download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CLAUDE



In [3]:
# ==============================================================================
# SCRIPT DE EXTRAÇÃO - SIM/DATASUS (Sistema de Informações sobre Mortalidade)
# UF: AC | Período: 2010-2023
# Grupos: Neoplasias (C, D) | Aparelho Circulatório (I)
# ==============================================================================

# --- 1. INSTALAÇÃO DE DEPENDÊNCIAS DE SISTEMA E PYTHON ---
!apt-get update -qq
!apt-get install -y -qq p7zip-full
!pip install pyreaddbc dbfread pandas -q

# --- IMPORTS ---
import os
import urllib.request
import pandas as pd
from dbfread import DBF
from pyreaddbc import dbc2dbf

# --- 2. PARÂMETROS DA PESQUISA ---
UF = "AC"
ANO_INICIO = 2010
ANO_FIM = 2023

GRUPOS_CID = {
    "Neoplasias": ["C", "D"],
    "Aparelho Circulatório": ["I"]
}

# Diretório temporário de trabalho
DIR_TEMP = "/content/temp_sim"
os.makedirs(DIR_TEMP, exist_ok=True)

# Lista que vai acumular os resultados consolidados
resultados = []

# --- 3. LOOP DE DOWNLOAD, CONVERSÃO E LEITURA ---
for ano in range(ANO_INICIO, ANO_FIM + 1):
    print(f"\n{'='*60}")
    print(f"Processando ano: {ano}")
    print(f"{'='*60}")

    nome_base = f"DO{UF}{ano}"
    caminho_dbc = os.path.join(DIR_TEMP, f"{nome_base}.dbc")
    caminho_dbf = os.path.join(DIR_TEMP, f"{nome_base}.dbf")

    # URLs de tentativa: maiúscula (.DBC) primeiro, minúscula (.dbc) depois
    url_maiuscula = f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/{nome_base}.DBC"
    url_minuscula = f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/{nome_base}.dbc"

    download_ok = False
    for url_tentativa in [url_maiuscula, url_minuscula]:
        try:
            print(f"Tentando baixar: {url_tentativa}")
            urllib.request.urlretrieve(url_tentativa, caminho_dbc)
            download_ok = True
            print("Download concluído com sucesso.")
            break
        except Exception as e:
            print(f"Falha no download ({url_tentativa}): {e}")

    if not download_ok:
        print(f"AVISO: Não foi possível baixar dados para o ano {ano}. Pulando...")
        continue

    # --- CONVERSÃO DBC -> DBF ---
    try:
        dbc2dbf(caminho_dbc, caminho_dbf)
        print("Conversão DBC -> DBF concluída.")
    except Exception as e:
        print(f"ERRO na conversão DBC->DBF para o ano {ano}: {e}")
        if os.path.exists(caminho_dbc):
            os.remove(caminho_dbc)
        continue

    # --- LEITURA DO DBF ---
    try:
        tabela = DBF(caminho_dbf, encoding="latin1")
        df_ano = pd.DataFrame(iter(tabela))
        print(f"Registros lidos: {len(df_ano)}")
    except Exception as e:
        print(f"ERRO na leitura do DBF para o ano {ano}: {e}")
        for f in [caminho_dbc, caminho_dbf]:
            if os.path.exists(f):
                os.remove(f)
        continue

    # --- 4. TRATAMENTO DA COLUNA E AGRUPAMENTO ---
    if "CAUSABAS" in df_ano.columns:
        df_ano["CAUSABAS"] = df_ano["CAUSABAS"].astype(str).str.strip().str.upper()

        for nome_grupo, prefixos in GRUPOS_CID.items():
            filtro = df_ano["CAUSABAS"].str.startswith(tuple(prefixos))
            total_obitos = int(filtro.sum())

            resultados.append({
                "ANO": ano,
                "GRUPO": nome_grupo,
                "OBITOS": total_obitos
            })

            print(f"  -> {nome_grupo}: {total_obitos} óbitos")
    else:
        print(f"AVISO: Coluna CAUSABAS não encontrada no ano {ano}.")

    # --- REMOÇÃO DOS ARQUIVOS TEMPORÁRIOS ---
    for f in [caminho_dbc, caminho_dbf]:
        if os.path.exists(f):
            os.remove(f)

    del df_ano

print(f"\n{'='*60}")
print("Processamento de todos os anos concluído.")
print(f"{'='*60}")

# --- 5. CONSOLIDAÇÃO E PADRONIZAÇÃO DA SAÍDA ---
df_final = pd.DataFrame(resultados, columns=["ANO", "GRUPO", "OBITOS"])
df_final = df_final.sort_values(by=["ANO", "GRUPO"]).reset_index(drop=True)

nome_arquivo_saida = f"mortalidade_{UF.lower()}_consolidado.csv"
df_final.to_csv(nome_arquivo_saida, index=False, encoding="utf-8-sig")

print(f"\nArquivo salvo: {nome_arquivo_saida}")
print(df_final.head(20))

# --- DOWNLOAD DO ARQUIVO ---
from google.colab import files
files.download(nome_arquivo_saida)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Processando ano: 2010
Tentando baixar: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2010.DBC
Download concluído com sucesso.
Conversão DBC -> DBF concluída.
Registros lidos: 3009
  -> Neoplasias: 353 óbitos
  -> Aparelho Circulatório: 606 óbitos

Processando ano: 2011
Tentando baixar: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2011.DBC
Download concluído com sucesso.
Conversão DBC -> DBF concluída.
Registros lidos: 3157
  -> Neoplasias: 399 óbitos
  -> Aparelho Circulatório: 715 óbitos

Processando ano: 2012
Tentando baixar: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2012.DBC
Download concluído com sucesso.
Conversão DBC -> DBF concluída.
Registros lidos: 3293
  -> Neoplasias: 497 óbitos
  -> Aparelho Circulatório: 659 óbitos

Processando ano

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gerar o gráfico


Atue como um especialista em visualização de dados de saúde pública e crie um script em Python para o Google Colab para gerar um gráfico de linhas interativo e profissional a partir do arquivo CSV gerado na etapa anterior.

Siga RIGOROSAMENTE os parâmetros e as regras de arquitetura fornecidas.

---
REGRAS OBRIGATÓRIAS DE ARQUITETURA DO CÓDIGO (NÃO MUDAR OU OMITIR):

1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS:
   - Instale e importe as bibliotecas necessárias para plotagem interativa e manipulação de dados:
     !pip install plotly pandas -q
   - Importe `pandas as pd` e `plotly.express as px` (ou `plotly.graph_objects as go`).

2. LEITURA FLEXÍVEL DO ARQUIVO CSV (CRÍTICO - ANTI-FALHA):
   - O script deve procurar o arquivo na pasta atual do Colab.
   - Faça uma busca dinâmica testando os arquivos que terminam com "_consolidado.csv" ou o padrão "mortalidade_ac_consolidado.csv".
   - Adicione uma checagem de segurança: se nenhum arquivo CSV for encontrado, mostre uma mensagem explicativa orientando o usuário a fazer o upload do arquivo CSV gerado na Etapa 1.

3. TRATAMENTO E PADRONIZAÇÃO DE DADOS:
   - Garanta que as colunas 'ANO' e 'OBITOS' sejam convertidas para tipo numérico (`pd.to_numeric`).
   - Garanta que os valores do eixe X (anos) sejam exibidos como NÚMEROS INTEIROS (ex: 2010, 2011, 2012) e NUNCA como decimais (ex: 2010.0 ou 2010.5).

4. DESIGN E ESTILIZAÇÃO DO GRÁFICO (QUALIDADE DE PUBLICAÇÃO):
   - Crie um gráfico de linhas mostrando a evolução temporal dos óbitos por GRUPO ("Neoplasias" vs "Aparelho Circulatório").
   - Destaque as linhas com marcadores de pontos em cada ano (`markers=True`).
   - Adicione um título claro informando o Estado e o Período analisado.
   - Configure o eixe X para mostrar todos os anos explicitamente (marcações ano a ano).
   - Use um tema visual limpo (`plotly_white` ou similar) com legenda visível e rótulos de eixos bem definidos (Ex: Eixo X: "Ano", Eixo Y: "Número de Óbitos").

5. EXIBIÇÃO E DOWNLOAD:
   - Exiba o gráfico interativo diretamente na tela do notebook Colab.
   - Salve o gráfico em formato HTML interativo (`mortalidade_grafico_interativo.html`) e adicione o comando `google.colab.files.download()` para baixar o arquivo no computador.
   - Entregue o código completo em um ÚNICO bloco executável.

In [5]:
# ==============================================================================
# SCRIPT DE VISUALIZAÇÃO - MORTALIDADE SIM/DATASUS
# Gráfico de Linhas Interativo (Plotly) - Evolução Temporal por Grupo de Causa
# ==============================================================================

# --- 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS ---
!pip install plotly pandas -q

import os
import glob
import pandas as pd
import plotly.express as px

# --- 2. LEITURA FLEXÍVEL DO ARQUIVO CSV (ANTI-FALHA) ---
pasta_atual = "/content"

# Busca dinâmica: primeiro tenta o padrão específico, depois qualquer *_consolidado.csv
candidatos = glob.glob(os.path.join(pasta_atual, "mortalidade_ac_consolidado.csv"))

if not candidatos:
    candidatos = glob.glob(os.path.join(pasta_atual, "*_consolidado.csv"))

if not candidatos:
    print("=" * 70)
    print("ARQUIVO NÃO ENCONTRADO!")
    print("=" * 70)
    print(
        "Nenhum arquivo terminado em '_consolidado.csv' foi localizado na pasta "
        f"atual ({pasta_atual}).\n\n"
        "AÇÃO NECESSÁRIA:\n"
        "1. Faça o upload do arquivo CSV gerado na Etapa 1 "
        "(ex: 'mortalidade_ac_consolidado.csv').\n"
        "2. No Colab, use o ícone de pasta (menu lateral esquerdo) > 'Upload' "
        "ou execute:\n\n"
        "   from google.colab import files\n"
        "   uploaded = files.upload()\n\n"
        "3. Após o upload, execute esta célula novamente."
    )
    raise FileNotFoundError("Nenhum arquivo '*_consolidado.csv' encontrado na pasta atual.")

caminho_csv = candidatos[0]
print(f"Arquivo encontrado: {caminho_csv}")

# --- 3. LEITURA E TRATAMENTO/PADRONIZAÇÃO DOS DADOS ---
df = pd.read_csv(caminho_csv)

# Garantir colunas numéricas
df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce")
df["OBITOS"] = pd.to_numeric(df["OBITOS"], errors="coerce")

# Remover eventuais linhas inválidas geradas pela conversão
df = df.dropna(subset=["ANO", "OBITOS"])

# Forçar ANO como inteiro (nunca decimal)
df["ANO"] = df["ANO"].astype(int)
df["OBITOS"] = df["OBITOS"].astype(int)

df = df.sort_values(by=["GRUPO", "ANO"]).reset_index(drop=True)

# --- Metadados para título dinâmico ---
ano_min = df["ANO"].min()
ano_max = df["ANO"].max()

# Tenta extrair a UF a partir do nome do arquivo (ex: mortalidade_ac_consolidado.csv -> AC)
nome_arquivo = os.path.basename(caminho_csv)
try:
    uf_extraida = nome_arquivo.split("_")[1].upper()
except IndexError:
    uf_extraida = "UF"

print(f"Período identificado: {ano_min} - {ano_max}")
print(f"UF identificada: {uf_extraida}")
print(df.head(10))

# --- 4. CRIAÇÃO DO GRÁFICO (QUALIDADE DE PUBLICAÇÃO) ---
fig = px.line(
    df,
    x="ANO",
    y="OBITOS",
    color="GRUPO",
    markers=True,
    template="plotly_white",
    title=f"Evolução da Mortalidade por Grupo de Causa CLAUDE- {uf_extraida} ({ano_min}-{ano_max})",
    labels={
        "ANO": "Ano",
        "OBITOS": "Número de Óbitos",
        "GRUPO": "Grupo de Causa"
    },
)

# Configuração explícita do eixo X: todos os anos, sem decimais
anos_unicos = sorted(df["ANO"].unique())

fig.update_xaxes(
    tickmode="array",
    tickvals=anos_unicos,
    ticktext=[str(a) for a in anos_unicos],
    tickangle=-45,
    title_text="Ano",
    showgrid=True,
    dtick=1
)

fig.update_yaxes(
    title_text="Número de Óbitos",
    showgrid=True,
    rangemode="tozero"
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.update_layout(
    title=dict(
        x=0.5,
        xanchor="center",
        font=dict(size=18)
    ),
    legend=dict(
        title="Grupo de Causa",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    font=dict(size=13),
    hovermode="x unified",
    margin=dict(t=100, b=80, l=60, r=40)
)

# --- 5. EXIBIÇÃO E DOWNLOAD ---
fig.show()

nome_html_saida = "mortalidade_grafico_interativo.html"
fig.write_html(nome_html_saida)
print(f"\nGráfico salvo em HTML: {nome_html_saida}")

from google.colab import files
files.download(nome_html_saida)

Arquivo encontrado: /content/mortalidade_ac_consolidado.csv
Período identificado: 2010 - 2023
UF identificada: AC
    ANO                  GRUPO  OBITOS
0  2010  Aparelho Circulatório     606
1  2011  Aparelho Circulatório     715
2  2012  Aparelho Circulatório     659
3  2013  Aparelho Circulatório     698
4  2014  Aparelho Circulatório     762
5  2015  Aparelho Circulatório     849
6  2016  Aparelho Circulatório     834
7  2017  Aparelho Circulatório     856
8  2018  Aparelho Circulatório     910
9  2019  Aparelho Circulatório     926



Gráfico salvo em HTML: mortalidade_grafico_interativo.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>